In [ ]:
import pandas as pd
import time
from pyspark.ml import Pipeline, PipelineModel
from pyspark.ml.classification import GBTClassifier, LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator, MulticlassClassificationEvaluator
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.sql import SparkSession
from pyspark.sql.functions import avg, col, count, floor, round as _round, sum as _sum, when
from sklearn.ensemble import (
    GradientBoostingClassifier as SklearnGradientBoostingClassifier,
    RandomForestClassifier as SklearnRandomForestClassifier,
)
from sklearn.linear_model import LogisticRegression as SklearnLogisticRegression
from sklearn.metrics import (
    average_precision_score, confusion_matrix, f1_score,
    precision_score, recall_score, roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline as SklearnPipeline
from sklearn.preprocessing import StandardScaler

In [2]:
spark = SparkSession.builder \
    .appName("PySpark_ML") \
    .master("local[*]") \
    .getOrCreate()
print(f"Spark Session успешно создана! Версия Spark: {spark.version}")

Spark Session успешно создана! Версия Spark: 3.5.1


# Разведочный анализ данных

Набор данных Credit Card Fraud Detection: транзакции по картам европейских держателей за сентябрь 2013 года.

Содержит следующие признаки:
- `V1`-`V28` — результат PCA-преобразования исходных признаков. По соображениям конфиденциальности их физический смысл скрыт.
- `Time` — секунды, прошедшие с момента первой транзакции в датасете.
- `Amount` — сумма транзакции.
- `Class` — целевая переменная: `1` — мошенничество, `0` — нормальная операция.

In [3]:
(df := spark.read.csv("data/creditcard.csv", header=True, inferSchema=True))

DataFrame[Time: double, V1: double, V2: double, V3: double, V4: double, V5: double, V6: double, V7: double, V8: double, V9: double, V10: double, V11: double, V12: double, V13: double, V14: double, V15: double, V16: double, V17: double, V18: double, V19: double, V20: double, V21: double, V22: double, V23: double, V24: double, V25: double, V26: double, V27: double, V28: double, Amount: double, Class: int]

In [4]:
df.printSchema()

root
 |-- Time: double (nullable = true)
 |-- V1: double (nullable = true)
 |-- V2: double (nullable = true)
 |-- V3: double (nullable = true)
 |-- V4: double (nullable = true)
 |-- V5: double (nullable = true)
 |-- V6: double (nullable = true)
 |-- V7: double (nullable = true)
 |-- V8: double (nullable = true)
 |-- V9: double (nullable = true)
 |-- V10: double (nullable = true)
 |-- V11: double (nullable = true)
 |-- V12: double (nullable = true)
 |-- V13: double (nullable = true)
 |-- V14: double (nullable = true)
 |-- V15: double (nullable = true)
 |-- V16: double (nullable = true)
 |-- V17: double (nullable = true)
 |-- V18: double (nullable = true)
 |-- V19: double (nullable = true)
 |-- V20: double (nullable = true)
 |-- V21: double (nullable = true)
 |-- V22: double (nullable = true)
 |-- V23: double (nullable = true)
 |-- V24: double (nullable = true)
 |-- V25: double (nullable = true)
 |-- V26: double (nullable = true)
 |-- V27: double (nullable = true)
 |-- V28: double (nulla

In [5]:
df.show(5)

+----+------------------+-------------------+----------------+------------------+-------------------+-------------------+-------------------+------------------+------------------+-------------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------+-------------------+------------------+-------------------+--------------------+-------------------+------------------+------------------+------------------+------------------+--------------------+-------------------+------+-----+
|Time|                V1|                 V2|              V3|                V4|                 V5|                 V6|                 V7|                V8|                V9|                V10|               V11|               V12|               V13|               V14|               V15|               V16|               V17|                V18|               V19|                V20|                 V21|                V22|     

In [6]:
print("Количество пропусков по колонкам:")
df.select([count(when(col(c).isNull(), c)).alias(c) for c in df.columns]).show()

Количество пропусков по колонкам:
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|Time| V1| V2| V3| V4| V5| V6| V7| V8| V9|V10|V11|V12|V13|V14|V15|V16|V17|V18|V19|V20|V21|V22|V23|V24|V25|V26|V27|V28|Amount|Class|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+
|   0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|  0|     0|    0|
+----+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+---+------+-----+



In [7]:
total = df.count()
distinct = df.distinct().count()
print(f"Всего строк: {total}, уникальных: {distinct}, дубликатов: {total - distinct}")

Всего строк: 284807, уникальных: 283726, дубликатов: 1081


In [8]:
df_duplicates = df.groupBy(df.columns).count().filter("count > 1")
print(f"Количество групп дубликатов: {df_duplicates.count()}")

Количество групп дубликатов: 773


In [9]:
print("Распределение групп дубликатов по классам:")
df_duplicates.groupBy("Class").count().show()

Распределение групп дубликатов по классам:
+-----+-----+
|Class|count|
+-----+-----+
|    1|   13|
|    0|  760|
+-----+-----+



In [10]:
print("Распределение дубликатов по классам:")
df_duplicates.withColumn("duplicates", col("Count") - 1) \
    .groupBy("Class").sum("duplicates").show()

Распределение дубликатов по классам:
+-----+---------------+
|Class|sum(duplicates)|
+-----+---------------+
|    1|             19|
|    0|           1062|
+-----+---------------+



In [11]:
df_clean = df.dropDuplicates()
print(f"Строк после удаления дубликатов: {df_clean.count()}")
print(f"Удалено строк: {df.count() - df_clean.count()}")

Строк после удаления дубликатов: 283726
Удалено строк: 1081


In [12]:
print("Распределение по классам без дубликатов:")
df_clean.groupBy("Class").count().orderBy("Class").show()

Распределение по классам без дубликатов:
+-----+------+
|Class| count|
+-----+------+
|    0|283253|
|    1|   473|
+-----+------+



In [13]:
print("Исходное распределение по классам:")
df.groupBy("Class").count().orderBy("Class").show()

Исходное распределение по классам:
+-----+------+
|Class| count|
+-----+------+
|    0|284315|
|    1|   492|
+-----+------+



In [14]:
df.select("Time", "Amount").describe().show()
df.select("V1", "V2", "V3", "V4", "V5").describe().show()

+-------+-----------------+------------------+
|summary|             Time|            Amount|
+-------+-----------------+------------------+
|  count|           284807|            284807|
|   mean|94813.85957508067| 88.34961925093077|
| stddev|47488.14595456596|250.12010924018867|
|    min|              0.0|               0.0|
|    max|         172792.0|          25691.16|
+-------+-----------------+------------------+

+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|summary|                  V1|                  V2|                  V3|                  V4|                  V5|
+-------+--------------------+--------------------+--------------------+--------------------+--------------------+
|  count|              284807|              284807|              284807|              284807|              284807|
|   mean|8.222932923226221...|7.983430022549729...|-1.34121624378835...|1.663746816699363...|8.749839304714502...|
|

In [15]:
quantiles_amount = [0.0, 0.25, 0.5, 0.75, 0.95, 0.97, 0.98, 0.985, 0.9875, 0.99, 1.0]
df_quantiles_amount = spark.createDataFrame(
    [(float(q), float(v)) for q, v in zip(
        quantiles_amount, df.approxQuantile("Amount", quantiles_amount, 0.01)
    )],
    ["Quantile", "Amount"]
)
df_quantiles_amount.orderBy("Quantile").show()

+--------+--------+
|Quantile|  Amount|
+--------+--------+
|     0.0|     0.0|
|    0.25|    5.44|
|     0.5|   21.04|
|    0.75|   74.95|
|    0.95|  318.15|
|    0.97|   436.2|
|    0.98|  527.53|
|   0.985|  613.35|
|  0.9875|  692.93|
|    0.99|25691.16|
|     1.0|25691.16|
+--------+--------+



In [16]:
quantiles_time = [0.0, 0.125, 0.25, 0.375, 0.5, 0.625, 0.75, 0.875, 1.0]
df_quantiles_time = spark.createDataFrame(
    [(float(q), float(v)) for q, v in zip(
        quantiles_time, df.approxQuantile("Time", quantiles_time, 0.01)
    )],
    ["Quantile", "Time"]
)
df_quantiles_time.orderBy("Quantile").show()

+--------+--------+
|Quantile|    Time|
+--------+--------+
|     0.0|     0.0|
|   0.125| 37570.0|
|    0.25| 53607.0|
|   0.375| 69259.0|
|     0.5| 83674.0|
|   0.625|123029.0|
|    0.75|139617.0|
|   0.875|154382.0|
|     1.0|172792.0|
+--------+--------+



In [17]:
print("Статистики Amount для нормальных транзакций:")
df.filter(col("Class") == 0).select("Amount").describe().show()

Статистики Amount для нормальных транзакций:
+-------+------------------+
|summary|            Amount|
+-------+------------------+
|  count|            284315|
|   mean| 88.29102242231271|
| stddev|250.10509222589235|
|    min|               0.0|
|    max|          25691.16|
+-------+------------------+



In [18]:
print("Статистики Amount для мошеннических транзакций:")
df.filter(col("Class") == 1).select("Amount").describe().show()

Статистики Amount для мошеннических транзакций:
+-------+-----------------+
|summary|           Amount|
+-------+-----------------+
|  count|              492|
|   mean|122.2113211382114|
| stddev|256.6832882977121|
|    min|              0.0|
|    max|          2125.87|
+-------+-----------------+



In [19]:
median_normal = df.filter(col("Class") == 0).approxQuantile("Amount", [0.5], 0.01)
median_fraud = df.filter(col("Class") == 1).approxQuantile("Amount", [0.5], 0.01)
print(f"Медиана Amount для нормальных транзакций: {median_normal[0]:.2f}")
print(f"Медиана Amount для мошеннических транзакций: {median_fraud[0]:.2f}")

Медиана Amount для нормальных транзакций: 21.49
Медиана Amount для мошеннических транзакций: 8.64


In [20]:
df_with_day = df.withColumn("Day", floor(col("Time") / 86400))
print("Распределение транзакций по дням:")
df_with_day.groupBy("Day").count().orderBy("Day").show()

Распределение транзакций по дням:
+---+------+
|Day| count|
+---+------+
|  0|144786|
|  1|140021|
+---+------+



In [21]:
print("Распределение мошеннических транзакций по дням:")
df_with_day.filter(col("Class") == 1).groupBy("Day").count().orderBy("Day").show()

Распределение мошеннических транзакций по дням:
+---+-----+
|Day|count|
+---+-----+
|  0|  281|
|  1|  211|
+---+-----+



In [22]:
df_time = df.withColumn("Day", floor(col("Time") / 86400)) \
            .withColumn("PartOfDayIndex", floor((col("Time") % 86400) / 21600))

df_time = df_time.withColumn(
    "PartOfDay",
    when(col("PartOfDayIndex") == 0, "Night")
    .when(col("PartOfDayIndex") == 1, "Morning")
    .when(col("PartOfDayIndex") == 2, "Day")
    .otherwise("Evening")
)

df_time_stats = df_time.groupBy("Day", "PartOfDay", "PartOfDayIndex").agg(
    count("*").alias("Total"),
    _sum(col("Class")).alias("Fraud"),
    _round(avg("Amount"), 2).alias("AvgAmount")
).withColumn(
    "FraudRate", _round(col("Fraud") / col("Total") * 100, 3)
).orderBy("Day", "PartOfDayIndex").drop("PartOfDayIndex")

print("Распределение транзакций по времени суток:")
df_time_stats.show(truncate=False)

Распределение транзакций по времени суток:
+---+---------+-----+-----+---------+---------+
|Day|PartOfDay|Total|Fraud|AvgAmount|FraudRate|
+---+---------+-----+-----+---------+---------+
|0  |Night    |12340|55   |61.86    |0.446    |
|0  |Morning  |35061|91   |102.08   |0.26     |
|0  |Day      |46850|71   |106.42   |0.152    |
|0  |Evening  |50535|64   |74.52    |0.127    |
|1  |Night    |11594|69   |60.6     |0.595    |
|1  |Morning  |35851|27   |94.51    |0.075    |
|1  |Day      |49585|63   |97.95    |0.127    |
|1  |Evening  |42991|52   |72.6     |0.121    |
+---+---------+-----+-----+---------+---------+



# Разбиение данных и создание Pipeline

In [23]:
df_fraud = df_clean.filter(col("Class") == 1)
df_normal = df_clean.filter(col("Class") == 0)

train_fraud, test_fraud = df_fraud.randomSplit([0.8, 0.2], seed=0)
train_normal, test_normal = df_normal.randomSplit([0.8, 0.2], seed=0)

train_df = train_fraud.union(train_normal)
test_df = test_fraud.union(test_normal)

def print_class_stats(title, df_temp):
    total = df_temp.count()
    fraud = df_temp.filter(col("Class") == 1).count()
    fraud_rate = fraud / total * 100
    print(
        f"{title}: {total} транзакций, мошеннических - {fraud} "
        f"({fraud_rate:.3f}%), нормальных - {total - fraud}"
    )

print_class_stats("Исходные данные", df_clean)
print_class_stats("Обучающие данные", train_df)
print_class_stats("Тестовые данные", test_df)

Исходные данные: 283726 транзакций, мошеннических - 473 (0.167%), нормальных - 283253
Обучающие данные: 227242 транзакций, мошеннических - 383 (0.169%), нормальных - 226859
Тестовые данные: 56484 транзакций, мошеннических - 90 (0.159%), нормальных - 56394


In [24]:
train_df.cache()
test_df.cache()
print(f"train_df: {train_df.count()} строк")
print(f"test_df: {test_df.count()} строк")

train_df: 226918 строк
test_df: 56808 строк


In [25]:
feature_cols = [f"V{i}" for i in range(1, 29)] + ["Time", "Amount"]

assembler_lr = VectorAssembler(inputCols=feature_cols, outputCol="features")
model_lr = LogisticRegression(
    featuresCol="features", labelCol="Class",
    maxIter=100, regParam=0.0, elasticNetParam=0.0,
    tol=1e-06, threshold=0.5, standardization=True,
)
pipeline_lr = Pipeline(stages=[assembler_lr, model_lr])

assembler_rf = VectorAssembler(inputCols=feature_cols, outputCol="features")
model_rf = RandomForestClassifier(
    featuresCol="features", labelCol="Class",
    maxDepth=5, minInstancesPerNode=1, impurity="gini",
    numTrees=20, bootstrap=True, seed=0,
)
pipeline_rf = Pipeline(stages=[assembler_rf, model_rf])

assembler_gbt = VectorAssembler(inputCols=feature_cols, outputCol="features")
model_gbt = GBTClassifier(
    featuresCol="features", labelCol="Class",
    maxDepth=5, minInstancesPerNode=1, impurity="variance",
    maxIter=20, lossType="logistic", stepSize=0.1, seed=0,
)
pipeline_gbt = Pipeline(stages=[assembler_gbt, model_gbt])

pipelines = {
    "LogisticRegression": pipeline_lr,
    "RandomForest": pipeline_rf,
    "GradientBoostedTrees": pipeline_gbt,
}

In [26]:
eval_auprc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderPR",
)
eval_roc_auc = BinaryClassificationEvaluator(
    labelCol="Class", rawPredictionCol="rawPrediction", metricName="areaUnderROC",
)

eval_precision_fraud = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction",
    metricName="precisionByLabel", metricLabel=1.0,
)
eval_recall_fraud = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction",
    metricName="recallByLabel", metricLabel=1.0,
)
eval_f1_fraud = MulticlassClassificationEvaluator(
    labelCol="Class", predictionCol="prediction",
    metricName="fMeasureByLabel", metricLabel=1.0,
)

def evaluate_model(model_name, preds):
    return {
        "Model": model_name,
        "AUPRC": eval_auprc.evaluate(preds),
        "ROC_AUC": eval_roc_auc.evaluate(preds),
        "Precision_fraud": eval_precision_fraud.evaluate(preds),
        "Recall_fraud": eval_recall_fraud.evaluate(preds),
        "F1_fraud": eval_f1_fraud.evaluate(preds),
    }

def show_confusion_matrix(model_name, preds):
    print(f"Матрица спутанности модели {model_name}:")
    preds.withColumn("prediction", col("prediction").cast("int")) \
        .groupBy("Class", "prediction").count().orderBy("Class", "prediction").show()

# Обучение моделей в составе Pipeline

In [ ]:
models = {}
preds_by_model = {}
results = []

for model_name, pipeline in pipelines.items():
    print(f"Обучение модели {model_name}:")
    t0 = time.time()
    model = pipeline.fit(train_df)
    t_fit = time.time() - t0
    print(f"- время обучения: {round(t_fit, 3)} сек.")

    t1 = time.time()
    preds = model.transform(test_df)
    preds.cache()
    _ = preds.count()
    t_pred = time.time() - t1
    print(f"- время предсказания: {round(t_pred, 3)} сек.")

    results.append(evaluate_model(model_name, preds))
    models[model_name] = model
    preds_by_model[model_name] = preds

Обучение модели LogisticRegression:
- время обучения: 41.083 сек.
- время предсказания: 2.064 сек.
Обучение модели RandomForest:
- время обучения: 13.272 сек.
- время предсказания: 1.816 сек.
Обучение модели GradientBoostedTrees:
- время обучения: 90.797 сек.
- время предсказания: 1.773 сек.


In [28]:
results_df = pd.DataFrame(results).set_index("Model")
results_df = results_df[["AUPRC", "ROC_AUC", "Precision_fraud", "Recall_fraud", "F1_fraud"]]
print(results_df.round(3).to_string())

                      AUPRC  ROC_AUC  Precision_fraud  Recall_fraud  F1_fraud
Model                                                                        
LogisticRegression    0.663    0.978            0.818         0.685     0.746
RandomForest          0.800    0.982            0.821         0.750     0.784
GradientBoostedTrees  0.831    0.981            0.897         0.761     0.824


In [29]:
for model_name, preds in preds_by_model.items():
    show_confusion_matrix(model_name, preds)

Матрица спутанности модели LogisticRegression:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56702|
|    0|         1|   14|
|    1|         0|   29|
|    1|         1|   63|
+-----+----------+-----+

Матрица спутанности модели RandomForest:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56701|
|    0|         1|   15|
|    1|         0|   23|
|    1|         1|   69|
+-----+----------+-----+

Матрица спутанности модели GradientBoostedTrees:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56708|
|    0|         1|    8|
|    1|         0|   22|
|    1|         1|   70|
+-----+----------+-----+



# Подбор значений гиперпараметров моделей

In [30]:
def cross_validate(model_name, pipeline, param_grid, evaluator, train):
    print(f"Обучение модели {model_name} с помощью CrossValidator:")
    cv = CrossValidator(
        estimator=pipeline,
        estimatorParamMaps=param_grid,
        evaluator=evaluator,
        numFolds=3,
        parallelism=1,
        seed=0,
    )
    t0 = time.time()
    cv_model = cv.fit(train)
    t_fit = time.time() - t0
    t_fit_m, t_fit_s = divmod(int(t_fit), 60)
    print(f"- время обучения: {t_fit_m} мин. {t_fit_s} сек.")

    param_maps = cv_model.getEstimatorParamMaps()
    avg_metrics = cv_model.avgMetrics

    print(f"Средние значения метрики для каждой комбинации:")
    for i, (pm, m) in enumerate(zip(param_maps, avg_metrics)):
        print(f"{i + 1}. AUPRC = {m:.4f} | {', '.join(f'{p.name}={v}' for p, v in pm.items())}")
    print(f"Лучшая комбинация: {max(range(len(avg_metrics)), key=lambda i: avg_metrics[i]) + 1}")

    return cv_model

In [31]:
model_lr_cv = LogisticRegression(featuresCol="features", labelCol="Class")
pipeline_lr_cv = Pipeline(stages=[
    VectorAssembler(inputCols=feature_cols, outputCol="features"), model_lr_cv,
])
grid_lr = ParamGridBuilder() \
    .addGrid(model_lr_cv.regParam, [0.0, 0.01, 0.1]) \
    .build()
cv_lr = cross_validate("LogisticRegression", pipeline_lr_cv, grid_lr, eval_auprc, train_df)

Обучение модели LogisticRegression с помощью CrossValidator:
- время обучения: 4 мин. 3 сек.
Средние значения метрики для каждой комбинации:
1. AUPRC = 0.7149 | regParam=0.0
2. AUPRC = 0.7177 | regParam=0.01
3. AUPRC = 0.7087 | regParam=0.1
Лучшая комбинация: 2


In [32]:
model_rf_cv = RandomForestClassifier(featuresCol="features", labelCol="Class", seed=0)
pipeline_rf_cv = Pipeline(stages=[
    VectorAssembler(inputCols=feature_cols, outputCol="features"), model_rf_cv,
])
grid_rf = ParamGridBuilder() \
    .addGrid(model_rf_cv.numTrees, [30, 50]) \
    .addGrid(model_rf_cv.maxDepth, [5, 8]) \
    .build()
cv_rf = cross_validate("RandomForest", pipeline_rf_cv, grid_rf, eval_auprc, train_df)

Обучение модели RandomForest с помощью CrossValidator:
- время обучения: 7 мин. 43 сек.
Средние значения метрики для каждой комбинации:
1. AUPRC = 0.7780 | numTrees=30, maxDepth=5
2. AUPRC = 0.7904 | numTrees=30, maxDepth=8
3. AUPRC = 0.7723 | numTrees=50, maxDepth=5
4. AUPRC = 0.8140 | numTrees=50, maxDepth=8
Лучшая комбинация: 4


In [33]:
model_gbt_cv = GBTClassifier(featuresCol="features", labelCol="Class", seed=0)
pipeline_gbt_cv = Pipeline(stages=[
    VectorAssembler(inputCols=feature_cols, outputCol="features"), model_gbt_cv,
])
grid_gbt = ParamGridBuilder() \
    .addGrid(model_gbt_cv.maxIter, [30, 50]) \
    .addGrid(model_gbt_cv.maxDepth, [5]) \
    .build()
cv_gbt = cross_validate("GradientBoostedTrees", pipeline_gbt_cv, grid_gbt, eval_auprc, train_df)

Обучение модели GradientBoostedTrees с помощью CrossValidator:
- время обучения: 28 мин. 28 сек.
Средние значения метрики для каждой комбинации:
1. AUPRC = 0.7767 | maxIter=30, maxDepth=5
2. AUPRC = 0.7772 | maxIter=50, maxDepth=5
Лучшая комбинация: 2


In [ ]:
cv_models = {
    "LogisticRegression": cv_lr,
    "RandomForest": cv_rf,
    "GradientBoostedTrees": cv_gbt,
}
preds_by_cv_model = {}
results_cv = []

for model_name, cv_model in cv_models.items():
    preds = cv_model.transform(test_df)
    preds.cache()
    _ = preds.count()
    results_cv.append(evaluate_model(model_name, preds))
    preds_by_cv_model[model_name] = preds

results_cv_df = pd.DataFrame(results_cv).set_index("Model")
results_cv_df = results_cv_df[["AUPRC", "ROC_AUC", "Precision_fraud", "Recall_fraud", "F1_fraud"]]
print(results_cv_df.round(3).to_string())

                      AUPRC  ROC_AUC  Precision_fraud  Recall_fraud  F1_fraud
Model                                                                        
LogisticRegression    0.661    0.978            0.755         0.402     0.525
RandomForest          0.870    0.989            0.900         0.783     0.837
GradientBoostedTrees  0.851    0.992            0.899         0.772     0.830


In [35]:
for model_name, preds in preds_by_cv_model.items():
    show_confusion_matrix(model_name, preds)

Матрица спутанности модели LogisticRegression:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56704|
|    0|         1|   12|
|    1|         0|   55|
|    1|         1|   37|
+-----+----------+-----+

Матрица спутанности модели RandomForest:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56708|
|    0|         1|    8|
|    1|         0|   20|
|    1|         1|   72|
+-----+----------+-----+

Матрица спутанности модели GradientBoostedTrees:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56708|
|    0|         1|    8|
|    1|         0|   21|
|    1|         1|   71|
+-----+----------+-----+



In [37]:
filenames = {
    "LogisticRegression": "pipeline_lr_best",
    "RandomForest": "pipeline_rf_best",
    "GradientBoostedTrees": "pipeline_gbt_best",
}

for model_name, cv_model in cv_models.items():
    path = f"models/{filenames[model_name]}"
    cv_model.bestModel.write().overwrite().save(path)
    print(f"Модель сохранена по пути: {path}")

Модель сохранена по пути: models/pipeline_lr_best
Модель сохранена по пути: models/pipeline_rf_best
Модель сохранена по пути: models/pipeline_gbt_best


# Проверка сохранённых Pipeline

In [ ]:
models_pyspark = {
    "LogisticRegression": PipelineModel.load("models/pipeline_lr_best"),
    "RandomForest": PipelineModel.load("models/pipeline_rf_best"),
    "GradientBoostedTrees": PipelineModel.load("models/pipeline_gbt_best"),
}
preds_by_pyspark_model = {}
results_pyspark = []

for model_name, model in models_pyspark.items():
    preds = model.transform(test_df)
    preds.cache()
    _ = preds.count()
    results_pyspark.append(evaluate_model(model_name, preds))
    preds_by_pyspark_model[model_name] = preds

results_pyspark_df = pd.DataFrame(results_pyspark).set_index("Model")
results_pyspark_df = results_pyspark_df[["AUPRC", "ROC_AUC", "Precision_fraud", "Recall_fraud", "F1_fraud"]]
print(results_pyspark_df.round(3).to_string())

                      AUPRC  ROC_AUC  Precision_fraud  Recall_fraud  F1_fraud
Model                                                                        
LogisticRegression    0.662    0.978            0.755         0.402     0.525
RandomForest          0.870    0.989            0.900         0.783     0.837
GradientBoostedTrees  0.851    0.992            0.899         0.772     0.830


In [40]:
for model_name, preds in preds_by_pyspark_model.items():
    show_confusion_matrix(model_name, preds)

Матрица спутанности модели LogisticRegression:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56704|
|    0|         1|   12|
|    1|         0|   55|
|    1|         1|   37|
+-----+----------+-----+

Матрица спутанности модели RandomForest:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56708|
|    0|         1|    8|
|    1|         0|   20|
|    1|         1|   72|
+-----+----------+-----+

Матрица спутанности модели GradientBoostedTrees:
+-----+----------+-----+
|Class|prediction|count|
+-----+----------+-----+
|    0|         0|56708|
|    0|         1|    8|
|    1|         0|   21|
|    1|         1|   71|
+-----+----------+-----+



# Эквивалентная реализация на sklearn

In [ ]:
df_fraud_mini  = df_fraud.sample(fraction=0.25, seed=0).toPandas()
df_normal_mini = df_normal.sample(fraction=0.25, seed=0).toPandas()
df_mini = pd.concat([df_fraud_mini, df_normal_mini], ignore_index=True)

X = df_mini[feature_cols].values
y = df_mini["Class"].values
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=0,
)

def print_class_stats_sklearn(title, df_temp):
    total = len(df_temp)
    fraud = df_temp.sum()
    fraud_rate = fraud / total * 100
    print(
        f"{title}: {total} транзакций, мошеннических - {fraud} "
        f"({fraud_rate:.3f}%), нормальных - {total - fraud}"
    )

print_class_stats_sklearn("Исходные данные", y)
print_class_stats_sklearn("Обучающие данные", y_train)
print_class_stats_sklearn("Тестовые данные", y_test)

Исходные данные: 70988 транзакций, мошеннических - 131 (0.185%), нормальных - 70857
Обучающие данные: 56790 транзакций, мошеннических - 105 (0.185%), нормальных - 56685
Тестовые данные: 14198 транзакций, мошеннических - 26 (0.183%), нормальных - 14172


In [44]:
def evaluate_model_sklearn(model_name, y_true, y_pred, y_proba):
    return {
        "Model": model_name,
        "AUPRC": average_precision_score(y_true, y_proba),
        "ROC_AUC": roc_auc_score(y_true, y_proba),
        "Precision_fraud": precision_score(y_true, y_pred, pos_label=1, zero_division=0),
        "Recall_fraud": recall_score(y_true, y_pred, pos_label=1, zero_division=0),
        "F1_fraud": f1_score(y_true, y_pred, pos_label=1, zero_division=0),
    }

def show_confusion_matrix_sklearn(model_name, y_test, y_pred):
    print(f"Матрица спутанности модели {model_name}:")
    print(confusion_matrix(y_test, y_pred, labels=[0, 1]))

models_sklearn = {
    "LogisticRegression": SklearnPipeline([
        ("scaler", StandardScaler()),
        ("clf", SklearnLogisticRegression(C=100, max_iter=100, random_state=0)),
    ]),
    "RandomForest": SklearnRandomForestClassifier(
        n_estimators=50, max_depth=8, random_state=0,
    ),
    "GradientBoostedTrees": SklearnGradientBoostingClassifier(
        n_estimators=50, max_depth=5, learning_rate=0.1, random_state=0,
    ),
}
preds_by_sklearn_model = {}
results_sklearn = []

for model_name, model in models_sklearn.items():
    print(f"Обучение модели {model_name}:")
    t0 = time.time()
    model.fit(X_train, y_train)
    t_fit = time.time() - t0
    print(f"- время обучения: {round(t_fit, 3)} сек.")

    t1 = time.time()
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    t_pred = time.time() - t1
    print(f"- время предсказания: {round(t_pred, 3)} сек.")

    results_sklearn.append(evaluate_model_sklearn(model_name, y_test, y_pred, y_proba))
    preds_by_sklearn_model[model_name] = y_pred

Обучение модели LogisticRegression:
- время обучения: 0.509 сек.
- время предсказания: 0.009 сек.
Обучение модели RandomForest:
- время обучения: 14.517 сек.
- время предсказания: 0.037 сек.
Обучение модели GradientBoostedTrees:
- время обучения: 84.986 сек.
- время предсказания: 0.013 сек.


In [45]:
results_sklearn_df = pd.DataFrame(results_sklearn).set_index("Model")
print(results_sklearn_df.round(3).to_string())

                      AUPRC  ROC_AUC  Precision_fraud  Recall_fraud  F1_fraud
Model                                                                        
LogisticRegression    0.788    0.948            0.792         0.731     0.760
RandomForest          0.855    0.966            0.880         0.846     0.863
GradientBoostedTrees  0.672    0.936            0.742         0.885     0.807


In [ ]:
for model_name, preds in preds_by_sklearn_model.items():
    show_confusion_matrix_sklearn(model_name, y_test, preds)

Матрица спутанности модели LogisticRegression:
[[14167     5]
 [    7    19]]
Матрица спутанности модели RandomForest:
[[14169     3]
 [    4    22]]
Матрица спутанности модели GradientBoostedTrees:
[[14164     8]
 [    3    23]]


In [57]:
spark_view = results_pyspark_df.copy()
spark_view = spark_view.reset_index()
spark_view["Model"] = spark_view["Model"] + " (Spark)"

sklearn_view = results_sklearn_df.copy()
sklearn_view = sklearn_view.reset_index()
sklearn_view["Model"] = sklearn_view["Model"] + " (sklearn)"

results_compare = pd.concat([spark_view, sklearn_view], ignore_index=True).set_index("Model")
results_compare = results_compare.loc[[
    "LogisticRegression (Spark)", "LogisticRegression (sklearn)",
    "RandomForest (Spark)", "RandomForest (sklearn)",
    "GradientBoostedTrees (Spark)", "GradientBoostedTrees (sklearn)",
]]
print(results_compare.round(3).to_string())

                                AUPRC  ROC_AUC  Precision_fraud  Recall_fraud  F1_fraud
Model                                                                                  
LogisticRegression (Spark)      0.662    0.978            0.755         0.402     0.525
LogisticRegression (sklearn)    0.788    0.948            0.792         0.731     0.760
RandomForest (Spark)            0.870    0.989            0.900         0.783     0.837
RandomForest (sklearn)          0.855    0.966            0.880         0.846     0.863
GradientBoostedTrees (Spark)    0.851    0.992            0.899         0.772     0.830
GradientBoostedTrees (sklearn)  0.672    0.936            0.742         0.885     0.807
